# Lab 14 · Xưởng kể chuyện: kim tự tháp, lời vừa số & audit

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành tuần 14**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo buổi 14 audit một "báo cáo AI" 5 kết luận. Lab này luyện chiều ngược
lại — **tự viết cho chuẩn** — rồi tự tay tạo một nghịch lý Simpson và audit một kết luận
mới. Tuần này cũng là **tuần nộp bài tập lớn**: 30 phút clinic cuối là buổi soát nộp.

## Cách làm việc trong buổi lab

- Bài tập được chia bước; mỗi bước có ô `TODO` và phần kiểm tra `assert` — chạy qua hết
  `assert` nghĩa là bạn làm đúng.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🔒 ở giờ lý thuyết đo đúng các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn 🔓: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Bạn kẹt quá 3 phút ở một bước: gọi trợ giảng.

## Mục tiêu

Sau buổi lab, bạn:

1. Sắp xếp một mục báo cáo theo cấu trúc kim tự tháp (kết luận → bằng chứng → giới hạn).
2. Phân biệt "lời vừa với số" và "lời quá cỡ" trên các cặp số–lời cụ thể.
3. Tự tính một nghịch lý Simpson: hai nhóm cùng tăng, tổng gộp vẫn giảm.
4. Audit một kết luận mới bằng đúng quy trình 4 bước.

### Bước 1 · Xếp kim tự tháp (~10 phút)

Bốn câu dưới đây thuộc một mục báo cáo về giá Santiago — nhưng bị xáo trộn.
Xếp lại theo **kim tự tháp**: (1) kết luận → (2) bằng chứng → (3) giới hạn →
(4) chi tiết phương pháp (đẩy xuống cuối/phụ lục).

In [ ]:
CAU = {
 "A": "Con số dựa trên 17.688 listing có giá hợp lệ của snapshot 29/06/2026; 846 listing không khai giá đã được loại và ghi trong qa_report.",
 "B": "Giá thuê điển hình ở Santiago là 59.000 CLP/đêm (~1,65 triệu đồng) — thấp hơn đáng kể ấn tượng '118 nghìn' nếu dùng trung bình.",
 "C": "Trung vị bền với outlier: 177 listing giá cực đoan (đã gắn cờ) kéo trung bình lên 44% nhưng gần như không đổi trung vị.",
 "D": "Kết luận chỉ áp dụng cho giá niêm yết một đêm, chưa gồm phí vệ sinh/dịch vụ; giá thực tế khách trả có thể cao hơn.",
}
for k, v in CAU.items():
    print(f"{k}. {v[:100]}…")

In [ ]:
# TODO: điền thứ tự 4 chữ cái theo kim tự tháp
thu_tu = [...]

# --- Ô kiểm tra ---
assert thu_tu == ["B", "C", "D", "A"], "Kết luận trước — bằng chứng — giới hạn — phương pháp sau cùng"
print("Chuẩn kim tự tháp: người đọc bận rộn dừng ở câu 1 vẫn nhận đúng thông điệp.")

### Bước 2 · Lời vừa với số (~10 phút)

Với mỗi cặp (số liệu → lời viết), gán `"vua"` (vừa cỡ) hoặc `"qua"` (quá cỡ).

In [ ]:
CAP = {
 1: ("YoY +53.7% (T5/2026 vs T5/2025)", "Thị trường tăng trưởng mạnh so với cùng kỳ"),
 2: ("+3.0% so với tháng trước", "Bứt phá thần tốc trong tháng 5"),
 3: ("Hệ số tương quan 0.081", "Nhiệt độ ngày ảnh hưởng rõ rệt tới lượng review"),
 4: ("Đúng 1 listing đòi ở tối thiểu 730 đêm", "Một vài ca cực đoan cần gắn cờ"),
 5: ("Nhóm es chiếm 65.4% (heuristic chưa kiểm định)", "Chắc chắn 2/3 khách nói tiếng Tây Ban Nha"),
}

# TODO: gán nhãn cho từng cặp
nhan = {1: ..., 2: ..., 3: ..., 4: ..., 5: ...}

# --- Ô kiểm tra ---
assert nhan == {1: "vua", 2: "qua", 3: "qua", 4: "vua", 5: "qua"}
print("Đúng cả 5 — từ ngữ là một phần của phép đo.")

Ba cặp "quá cỡ" đều là số bạn đã tự tính trong các lab trước: +3% (lab 8) không phải
"bứt phá"; 0.081 (lab 6) là "gần như không liên hệ"; 65.4% (lab 7) là heuristic chưa đo
chất lượng — "chắc chắn" là từ bị cấm ở đó.

### Bước 3 · Tự tạo một nghịch lý Simpson (~12 phút)

Hai phân khúc **cùng tăng giá 10%**, nhưng tổng gộp lại giảm. Không tin? Tự tính.

In [ ]:
import pandas as pd

bang = pd.DataFrame({
    "ky":        ["T9",  "T9",  "T6",  "T6"],
    "phan_khuc": ["cao", "re",  "cao", "re"],
    "n":         [100,   100,   80,    220],
    "gia_tb":    [60.0,  30.0,  66.0,  33.0],   # nghìn CLP — mỗi phân khúc +10%
})

# TODO: tính giá trung bình GỘP của từng kỳ:
#       tổng(n × gia_tb) / tổng(n) cho mỗi kỳ (groupby "ky" rồi apply công thức,
#       hoặc tính tay hai dòng)
gop_t9 = ...
gop_t6 = ...

# --- Ô kiểm tra ---
assert gop_t9 == 45.0 and gop_t6 == 41.8
print(f"Mỗi phân khúc +10%, nhưng gộp: {gop_t9} → {gop_t6} (GIẢM {(1 - gop_t6/gop_t9):.0%}).")

Thủ phạm: **cơ cấu mẫu đổi** — phân khúc rẻ phình từ 50% lên 73% số listing, kéo
trung bình gộp tụt dù mọi phân khúc đều tăng. Bài tập lớn so 2 snapshot *bắt buộc* phải
hỏi "mix có đổi không?" trước khi tin bất kỳ con số gộp nào.

### Bước 4 · Audit một kết luận mới (~15 phút)

Kết luận (chưa có trong notebook demo): *"Host chuyên nghiệp **được khách yêu thích hơn**
host cá nhân — 15,5 so với 12,7 review/năm."* Audit 4 bước:

In [ ]:
URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/listings.csv")
ds = pd.read_csv(URL)
ds["chuyen"] = ds["calculated_host_listings_count"] >= 5

# Bước 1 — truy số: TODO tính lại 2 con số 15.5 và 12.7
kiem = ...    # groupby "chuyen", mean của number_of_reviews_ltm, round 1

# --- Ô kiểm tra bước 1 ---
assert round(kiem[True], 1) == 15.5 and round(kiem[False], 1) == 12.7
print("Bước 1 ✓ — số đúng. Nhưng số đúng chưa phải kết luận đúng…")

In [ ]:
# Bước 2 (phương pháp) + Bước 3 (diễn giải): chọn MỘT phán quyết đúng nhất
# A. "Giữ nguyên — số đúng thì kết luận đúng."
# B. "Hạ cấp lời: số đo là LƯỢNG REVIEW (proxy của lượng đặt phòng), không đo
#     'yêu thích'; muốn nói yêu thích phải dùng điểm đánh giá — viết lại thành
#     'được đặt nhiều hơn' và kiểm thêm cắt lớp."
# C. "Bác bỏ — số liệu sai."
phan_quyet = "..."

# --- Ô kiểm tra ---
assert phan_quyet == "B"
print("Chuẩn: 'review nhiều' đo mức ĐƯỢC ĐẶT, không đo mức YÊU THÍCH — proxy phải gọi đúng tên.")

Đây là kiểu lỗi khó bắt nhất — mọi con số đều đúng, chỉ **một chữ** ("yêu thích")
vượt quá thứ dữ liệu đo được. Vấn đáp tuần sau sẽ hỏi đúng những câu như vậy vào
báo cáo của nhóm bạn.

## Bài tự làm 🔓 · Audit chéo bản nháp của nhóm

Lấy bản nháp báo cáo của nhóm (hoặc một mục bất kỳ đã viết): (1) chạy audit 4 bước lên
**2 kết luận lớn nhất**; (2) soát từng câu bằng phép thử "lời vừa số" của Bước 2;
(3) mọi thứ AI đã viết hộ trong báo cáo — đối chiếu mục "AI sai ở đâu" trong
`AI_USAGE.md` (nếu mục đó đang trống mà nhóm có dùng AI: đó chính là việc phải làm
trước khi nộp).

In [ ]:
# Ghi chép audit chéo của bạn ở đây (markdown/ô text tự do)

---

## 🧭 BTL clinic tuần 14 — SOÁT NỘP (~30 phút, quan trọng nhất kỳ)

Hạn nộp: **23:59 Chủ nhật tuần này** (ngày cụ thể: Canvas Portal). Soát theo checklist
nộp của deck buổi 15 — dưới đây là bản rút gọn để tự chấm tại chỗ:

1. ☐ `run_pipeline.py`/Makefile: **một lệnh** chạy từ raw đến mọi kết quả; thành phố +
   snapshot nằm trong `configs/`, không hard-code.
2. ☐ Cache LLM đầy đủ — `--skip-llm` chạy **không cần API key**; key không xuất hiện
   ở bất kỳ đâu trong repo (grep cả lịch sử commit).
3. ☐ `data/samples/` có bộ ≥100 nhãn tay; `reports/` có PDF + `qa_report.csv` +
   `slides.pdf`; `figures/` sinh tự động.
4. ☐ `AI_USAGE.md` cập nhật lần cuối — mục "AI sai ở đâu" có ví dụ thật.
5. ☐ **Bài kiểm tra máy sạch** đã lên lịch (một thành viên clone mới, làm theo README) —
   muộn nhất thứ Bảy; tag `final` + ghi commit hash vào báo cáo.

> TA đi từng nhóm: hỏi "mục nào chưa ✅ và ai xử lý trước thứ Bảy?" — ghi sổ nhóm có
> nguy cơ trễ để báo giảng viên ngay sau buổi.

## Tóm tắt buổi lab

| Bạn đã làm | Dùng cho |
|---|---|
| Kim tự tháp: kết luận trước, phương pháp sau | mọi mục của báo cáo nộp tuần này |
| 5 cặp số–lời: vừa hay quá cỡ | tự soát từng câu trước khi nộp |
| Simpson tự tính: mix đổi kéo gộp ngược chiều | so sánh 2 snapshot trong báo cáo |
| Audit "yêu thích vs được đặt" | vấn đáp tuần sau hỏi đúng kiểu này |

Tuần sau: **vấn đáp** — ôn theo deck buổi 15 + kế hoạch ôn 1 tuần trong đó. Chúc các nhóm
nộp suôn sẻ.